# Test model loading ...

## - Before building GradCAM or Streamlit, confirm the model loads correctly.

In [1]:

import os
print(os.getcwd())


C:\Users\rohit\Projects\chest_xray_disease_detection\notebooks


In [2]:

import os
os.chdir("..")   # adjust until you reach the repo root
print(os.getcwd())  # Check again


C:\Users\rohit\Projects\chest_xray_disease_detection


In [3]:

# Load The Model (saved_models/chest_xray_model.keras)

import tensorflow as tf

model = tf.keras.models.load_model("saved_models/chest_xray_model.keras")

print("Model loaded successfully")
model.summary()


Model loaded successfully


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)       ┃ Output Shape     ┃   Param # ┃ Connected to     ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ input_layer        │ (None, 224, 224, │         0 │ -                │
│ (InputLayer)       │ 3)               │           │                  │
├────────────────────┼──────────────────┼───────────┼──────────────────┤
│ zero_padding2d     │ (None, 230, 230, │         0 │ input_layer[0][… │
│ (ZeroPadding2D)    │ 3)               │           │                  │
├────────────────────┼──────────────────┼───────────┼──────────────────┤
│ conv1_conv         │ (None, 112, 112, │     9,408 │ zero_padding2d[… │
│ (Conv2D)           │ 64)              │           │                  │
├────────────────────┼──────────────────┼───────────┼──────────────────┤
│ conv1_bn           │ (None, 112, 112, │       256 │ conv1_conv[0][0] │
│ (BatchNormalizati… │ 64)              │           │                  │
├────────────────────┼──────────────────┼───────────┼──────────────────┤
│ conv1_relu         │ (None, 112, 112, │         0 │ conv1_bn[0][0]   │
│ (Activation)       │ 64)              │           │                  │
├────────────────────┼──────────────────┼───────────┼──────────────────┤
│ zero_padding2d_1   │ (None, 114, 114, │         0 │ conv1_relu[0][0] │
│ (ZeroPadding2D)    │ 64)              │           │                  │
├────────────────────┼──────────────────┼───────────┼──────────────────┤
│ pool1              │ (None, 56, 56,   │         0 │ zero_padding2d_… │
│ (MaxPooling2D)     │ 64)              │           │                  │
├────────────────────┼──────────────────┼───────────┼──────────────────┤
│ conv2_block1_0_bn  │ (None, 56, 56,   │       256 │ pool1[0][0]      │
│ (BatchNormalizati… │ 64)              │           │                  │
├────────────────────┼──────────────────┼───────────┼──────────────────┤
│ conv2_block1_0_re… │ (None, 56, 56,   │         0 │ conv2_block1_0_… │
│ (Activation)       │ 64)              │           │                  │
├────────────────────┼──────────────────┼───────────┼──────────────────┤
│ conv2_block1_1_co… │ (None, 56, 56,   │     8,192 │ conv2_block1_0_… │
│ (Conv2D)           │ 128)             │           │                  │
├────────────────────┼──────────────────┼───────────┼──────────────────┤
│ conv2_block1_1_bn  │ (None, 56, 56,   │       512 │ conv2_block1_1_… │
│ (BatchNormalizati… │ 128)             │           │                  │
├────────────────────┼──────────────────┼───────────┼──────────────────┤
│ conv2_block1_1_re… │ (None, 56, 56,   │         0 │ conv2_block1_1_… │
│ (Activation)       │ 128)             │           │                  │
├────────────────────┼──────────────────┼───────────┼──────────────────┤
│ conv2_block1_2_co… │ (None, 56, 56,   │    36,864 │ conv2_block1_1_… │
│ (Conv2D)           │ 32)              │           │                  │
├────────────────────┼──────────────────┼───────────┼──────────────────┤
│ conv2_block1_conc… │ (None, 56, 56,   │         0 │ pool1[0][0],     │
│ (Concatenate)      │ 96)              │           │ conv2_block1_2_… │
├────────────────────┼──────────────────┼───────────┼──────────────────┤
│ conv2_block2_0_bn  │ (None, 56, 56,   │       384 │ conv2_block1_co… │
│ (BatchNormalizati… │ 96)              │           │                  │
├────────────────────┼──────────────────┼───────────┼──────────────────┤
│ conv2_block2_0_re… │ (None, 56, 56,   │         0 │ conv2_block2_0_… │
│ (Activation)       │ 96)              │           │                  │
├────────────────────┼──────────────────┼───────────┼──────────────────┤
│ conv2_block2_1_co… │ (None, 56, 56,   │    12,288 │ conv2_block2_0_… │
│ (Conv2D)           │ 128)             │           │                  │
├────────────────────┼──────────────────┼───────────┼──────────────────┤
│ conv2_block2_1_bn  │ (None, 56, 56,   │       512 │ conv

 Total params: 7,088,748 (27.04 MB)

 Trainable params: 16,398 (64.05 KB)

 Non-trainable params: 7,039,552 (26.85 MB)

 Optimizer params: 32,798 (128.12 KB)

In [4]:

# Test a single image prediction ...

# Before explainability, make sure inference works ...

import cv2
import numpy as np

img_path = "data/raw/nih_sample/images/00000013_005.png"  # example

img = cv2.imread(img_path)
img = cv2.resize(img, (224, 224))
img = img / 255.0
img = np.expand_dims(img, axis=0)

pred = model.predict(img)

print(pred)


1/1 ━━━━━━━━━━━━━━━━━━━━ 16s 16s/step
[[6.7199931e-02 4.0114470e-02 2.2815302e-01 1.4078468e-01 1.1896365e-02
  3.6515955e-02 9.7676879e-03 1.7398225e-01 2.1088600e-02 7.3096673e-03
  2.6874480e-01 1.0050749e-03 4.2521805e-02 1.0742606e-04]]



## As this works ...

### The inference pipeline is valid.

In [5]:

# Check model input shape ...

model.input_shape


(None, 224, 224, 3)


## Identify the last convolution layer...
### It is required for GradCAM
    

In [6]:

# Identify the last convolution layer ...

for layer in model.layers:
    print(layer.name)


input_layer
zero_padding2d
conv1_conv
conv1_bn
conv1_relu
zero_padding2d_1
pool1
conv2_block1_0_bn
conv2_block1_0_relu
conv2_block1_1_conv
conv2_block1_1_bn
conv2_block1_1_relu
conv2_block1_2_conv
conv2_block1_concat
conv2_block2_0_bn
conv2_block2_0_relu
conv2_block2_1_conv
conv2_block2_1_bn
conv2_block2_1_relu
conv2_block2_2_conv
conv2_block2_concat
conv2_block3_0_bn
conv2_block3_0_relu
conv2_block3_1_conv
conv2_block3_1_bn
conv2_block3_1_relu
conv2_block3_2_conv
conv2_block3_concat
conv2_block4_0_bn
conv2_block4_0_relu
conv2_block4_1_conv
conv2_block4_1_bn
conv2_block4_1_relu
conv2_block4_2_conv
conv2_block4_concat
conv2_block5_0_bn
conv2_block5_0_relu
conv2_block5_1_conv
conv2_block5_1_bn
conv2_block5_1_relu
conv2_block5_2_conv
conv2_block5_concat
conv2_block6_0_bn
conv2_block6_0_relu
conv2_block6_1_conv
conv2_block6_1_bn
conv2_block6_1_relu
conv2_block6_2_conv
conv2_block6_concat
pool2_bn
pool2_relu
pool2_conv
pool2_pool
conv3_block1_0_bn
conv3_block1_0_relu
conv3_block1_1_conv
con